# Churn Prediction — Logistic Regression & Random Forest

Predicts which customers are likely to churn based on their RFM features.

**Prerequisite:** Run `1_customer_segmentation.ipynb` first to generate `data/customer_segments.csv`.

## 1. Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('data/customer_segments.csv')
print(df.shape)
df.head()

## 2. Define Churn

In [ ]:
# Customers who haven't purchased in >90 days are considered churned
df['Churn'] = np.where(df['Recency'] > 90, 1, 0)

# Average days between orders — a proxy for purchase cadence
df['Avg_between_orders'] = df['loyalty'] / df['Frequency']

print(f"Churned customers: {df['Churn'].sum()} ({df['Churn'].mean()*100:.1f}%)")
df.head()

## 3. Prepare Features

In [ ]:
# Drop non-predictive and target-leaking columns
X = df.drop(columns=['Customer ID', 'MinInvoiceDate', 'MaxInvoiceDate',
                     'Recency', 'Cluster', 'Cluster Name', 'Churn'])
y = df['Churn']

print(f"Features: {list(X.columns)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 4. Logistic Regression

In [ ]:
# class_weight='balanced' handles the imbalanced churn/non-churn ratio
model_lr = LogisticRegression(class_weight='balanced', max_iter=1000)
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

print("Logistic Regression")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(classification_report(y_test, y_pred_lr))

## 5. Random Forest

In [ ]:
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print("Random Forest")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
print(classification_report(y_test, y_pred_rf))

## 6. Feature Importance (Random Forest)

In [ ]:
feature_importance = pd.Series(
    model_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feature_importance.plot(kind='barh')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()